In [1]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re
import json

RUC_PRUEBA = "20607524476"
URL_SUNAT = "https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/FrameCriterioBusquedaWeb.jsp"

def limpiar(texto: str) -> str:
    if not texto:
        return ""
    return re.sub(r"\s+", " ", texto).strip()

async def consultar_sunat_completo(ruc: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        print("1. Conectando a SUNAT...")
        await page.goto(URL_SUNAT, wait_until="domcontentloaded", timeout=30000)

        # Ubicar input y escribir RUC
        input_sel = "input#txtRuc, input[name='search1'], input[placeholder*='RUC']"
        await page.wait_for_selector(input_sel, timeout=10000)
        await page.fill(input_sel, ruc)

        # Clic en Buscar
        print("2. Enviando consulta...")
        boton_buscar = "button#btnAceptar, button:has-text('Buscar'), input[value='Buscar']"
        await page.click(boton_buscar)

        # Esperar carga del resultado
        await page.wait_for_selector("text=Número de RUC:", timeout=15000)
        print("3. Ficha principal cargada.")

        # --- EXTRACCIÓN DINÁMICA DE LA FICHA COMPLETA ---
        html_ficha = await page.content()
        soup_ficha = BeautifulSoup(html_ficha, "html.parser")

        ficha_completa = {}

        # Mapeo de filas en las tablas de resultados
        for tr in soup_ficha.find_all("tr"):
            celdas = tr.find_all(["td", "th"])
            i = 0
            while i < len(celdas):
                texto_col = limpiar(celdas[i].get_text())
                # Si la celda actual es una etiqueta que termina en dos puntos ':'
                if texto_col.endswith(":") and (i + 1) < len(celdas):
                    clave = texto_col.rstrip(":")
                    # Verificar si la celda siguiente contiene múltiples párrafos o líneas (e.g. Actividades Económicas)
                    parrafos = [limpiar(p.get_text()) for p in celdas[i + 1].find_all(["p", "div"]) if limpiar(p.get_text())]
                    if len(parrafos) > 1:
                        valor = parrafos
                    else:
                        valor = limpiar(celdas[i + 1].get_text())
                    
                    ficha_completa[clave] = valor
                    i += 2
                else:
                    i += 1

        # Fallback para divs con estructura Bootstrap (list-group-item)
        for item in soup_ficha.find_all("div", class_=re.compile(r"list-group-item")):
            t = limpiar(item.get_text())
            if ":" in t:
                k, v = t.split(":", 1)
                k = limpiar(k)
                if k not in ficha_completa:
                    ficha_completa[k] = limpiar(v)

        # --- EXTRACCIÓN DE REPRESENTANTES LEGALES ---
        representantes = []
        boton_repleg = "button:has-text('Representante(s) Legal(es)'), input[value*='Representante']"

        if await page.query_selector(boton_repleg):
            print("4. Extrayendo Representantes Legales...")
            try:
                # Caso A: El botón abre un popup (nueva pestaña)
                async with context.expect_page(timeout=5000) as popup_info:
                    await page.click(boton_repleg)
                popup = await popup_info.value
                await popup.wait_for_load_state("domcontentloaded")
                html_rep = await popup.content()
                await popup.close()
            except Exception:
                # Caso B: Carga en la misma página o en un modal
                await page.wait_for_timeout(2000)
                html_rep = await page.content()

            soup_rep = BeautifulSoup(html_rep, "html.parser")
            tabla_rep = soup_rep.find("table", class_=re.compile(r"form-table|table")) or soup_rep.find("table")
            
            if tabla_rep:
                for row in tabla_rep.find_all("tr"):
                    cols = [limpiar(td.get_text()) for td in row.find_all("td")]
                    # Cabecera típica: Documento | Nro | Nombre | Cargo | Fecha Desde
                    if len(cols) >= 4 and not cols[0].lower().startswith("tipo"):
                        representantes.append({
                            "tipo_documento": cols[0],
                            "numero_documento": cols[1],
                            "nombre_completo": cols[2],
                            "cargo": cols[3],
                            "fecha_desde": cols[4] if len(cols) >= 5 else None
                        })

        await browser.close()
        return {
            "ficha": ficha_completa,
            "representantes": representantes
        }

# Ejecutar en el notebook
resultado = await consultar_sunat_completo(RUC_PRUEBA)

print("\n================== FICHA COMPLETA EXTRAÍDA ==================")
print(json.dumps(resultado["ficha"], indent=2, ensure_ascii=False))

print("\n================= REPRESENTANTES LEGALES ===================")
if resultado["representantes"]:
    for r in resultado["representantes"]:
        print(f"- [{r['cargo']}] {r['nombre_completo']} ({r['tipo_documento']} {r['numero_documento']}) - Desde: {r['fecha_desde']}")
else:
    print("No se encontraron representantes registrados para este RUC.")

1. Conectando a SUNAT...
2. Enviando consulta...
3. Ficha principal cargada.
4. Extrayendo Representantes Legales...

================== FICHA COMPLETA EXTRAÍDA ==================
{
  "IMPORTANTE": "Los comprobantes de pago o notas de débito emitidos por este contribuyente no dan derecho a crédito fiscal del IGV, en tanto se encuentra con estado de BAJA DE OFICIO",
  "Número de RUC": "20607524476 - INVERSIONES Y SERVICIOS HUAYNA DEL PERU EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA",
  "Tipo Contribuyente": "EMPRESA INDIVIDUAL DE RESP. LTDA",
  "Nombre Comercial": "INVERSIONES HUAYNA DEL PERU",
  "Fecha de Inscripción": "24/02/2021 Fecha de Inicio de Actividades: 15/03/2021",
  "Estado del Contribuyente": "BAJA DE OFICIO Fecha de Baja: 30/04/2026",
  "Condición del Contribuyente": "HABIDO",
  "Domicilio Fiscal": "CAL.VILLA LA RINCONADA LOTE. H-1 APV. VILLA LA RINCONADA CUSCO - CUSCO - SAN JERONIMO",
  "Sistema Emisión de Comprobante": "COMPUTARIZADO Actividad Comercio Exterior: SIN A